# The Connecting Narrative: Modules 12–15 as One Unified Framework
## How Diffusion, Bayesian Uncertainty, RL, and Self-Supervised Learning Are the Same Idea

## TL;DR — Plain English

**The secret:** Every algorithm in Modules 12–15 is answering the same question from a different angle:
> *"How do I generate molecules with desired properties while knowing what I don't know?"*

- **Module 12 (Diffusion):** How to generate diverse, realistic structures
- **Module 13 (Bayesian):** How to quantify what the model doesn't know
- **Module 14 (RL):** How to steer generation toward desired properties
- **Module 15 (SSL):** How to learn structure from unlabeled data

**After this notebook:** you'll see these four as one integrated system, not four separate topics.


## 🟢 Beginner Frame — Start Here If You Feel Overwhelmed

If you just finished Modules 12–15 and feel overwhelmed, this notebook shows you **one picture that makes everything click**. You don't need to memorize — you need to see the pattern.

Here is what each module is *really* doing:

| Module | One-line summary |
|--------|------------------|
| 12 — Diffusion | Learns to reverse noise → generates new structures |
| 13 — Bayesian  | Puts error bars on every prediction |
| 14 — RL        | Steers generation toward a reward signal |
| 15 — SSL       | Learns representations from unlabelled sequences |

All four share one loop: **generate → evaluate → update**. Every section below is just one camera angle on that loop.

> **Tip:** Read the unified framework diagram first, then come back and read the code cells. The diagram IS the mental model.

## The Unified Framework

```
PRE-TRAINING (Module 15)
  ESM-2 / BERT MLM on 250M sequences
  → learns protein "language" (what sequences are plausible)
  → provides embeddings used by EVERY other module

         ↓

GENERATION (Module 12)
  Diffusion / Flow Matching
  → samples from learned protein distribution
  → conditioned on ESM-2 embeddings for sequence-aware design
  → produces candidate structures/sequences

         ↓

UNCERTAINTY (Module 13)
  Bayesian inference / conformal prediction
  → estimates which predictions to trust
  → guides where to sample next (acquisition function)
  → prevents overconfident experimental suggestions

         ↓

STEERING (Module 14)
  GFlowNets / PPO / Bayesian Optimization
  → uses reward signal (ΔΔG, docking score, pLDDT)
  → updates generation to produce higher-reward candidates
  → Bayesian Optimization = data-efficient RL (use when oracle is expensive)

         ↓

ORACLE (Modules 07 + 10)
  AF3 + ΔΔG model
  → evaluates candidates
  → provides reward signal back to RL
  → loop continues until convergence
```


## 🟢 Complete Beginner's Guide

**Assumed background:** Has completed modules 12–15 individually. Now connecting them into a unified view.

### What this notebook gives you

After studying four separate modules on generative models, self-supervised learning, reinforcement learning, and Bayesian methods, this notebook shows you **why these are not separate topics** — they are four angles on the same problem: 'how do we learn useful representations of protein space and search it intelligently?'

### The unified view in plain English

- **Diffusion models (Module 12)** — learn the data distribution P(structure) and sample new structures from it.
- **Self-supervised learning (Module 15)** — learn representations without labels by solving pretext tasks (masked language modeling, contrastive objectives).
- **Reinforcement learning (Module 14)** — optimize sequences or structures toward a reward signal (e.g. predicted binding affinity).
- **Bayesian methods (Module 13)** — quantify uncertainty in predictions so you know when to trust the model.

### Vocabulary you MUST know

| Term | One-line definition |
|------|--------------------|
| `latent space` | Compressed representation of data learned by an encoder |
| `score function` | Gradient of log probability with respect to data (used in diffusion) |
| `contrastive loss` | Loss that pulls similar examples together, pushes dissimilar apart |
| `policy` | In RL, the function that maps state to action |
| `posterior` | In Bayes, the updated belief after seeing data |

### 3-Step Reading Strategy for Beginners

1. **Read all markdown cells first** — focus on the connection diagram between methods.
2. **Run code cells one at a time** — trace how a representation from module 15 feeds into module 14's policy.
3. **Modify one connection and re-run** — swap the representation method and compare downstream performance.

### If you're stuck

- **YouTube:** 'Generative Models Overview' by Lilian Weng (blog post companion: https://lilianweng.github.io/posts/2021-07-11-diffusion-models/)
- **YouTube:** 'Self-Supervised Learning' by Yannic Kilcher.
- **Book:** *Probabilistic Machine Learning* by Kevin Murphy, Part IV (Generative Models).
- **Web:** https://www.biorxiv.org/content/10.1101/2022.12.09.519842 — RFdiffusion paper.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ─────────────────────────────────────────────────────────────────────
# INTEGRATED PROTEIN DESIGN LOOP
# Shows how Modules 12-15 work together in one pipeline
# ─────────────────────────────────────────────────────────────────────

def simulate_ssl_embeddings(sequences: list, embed_dim: int = 64) -> np.ndarray:
    # Module 15: SSL pre-training produces embeddings
    # Real: ESM-2 last hidden layer (dim=1280 for 650M model)
    np.random.seed(42)
    embeddings = []
    for seq in sequences:
        # Simulate: similar sequences get similar embeddings
        base = np.random.randn(embed_dim)
        noise = np.random.randn(embed_dim) * 0.1
        embeddings.append(base + noise)
    return np.array(embeddings)

def diffusion_sample(embedding: np.ndarray, n_samples: int = 5) -> np.ndarray:
    # Module 12: Diffusion conditioned on SSL embedding
    # Real: RFdiffusion uses ESM-2 embeddings as conditioning
    np.random.seed(123)
    samples = []
    for _ in range(n_samples):
        # Sample structure (simplified as 2D coordinate)
        # Conditioning on embedding biases samples toward high-probability region
        x = embedding[:2] * 0.5 + np.random.randn(2) * 0.8
        samples.append(x)
    return np.array(samples)

def bayesian_uncertainty(candidates: np.ndarray, model_noise: float = 0.3) -> tuple:
    # Module 13: Estimate epistemic + aleatoric uncertainty
    # Real: bootstrap ensemble or MC Dropout
    np.random.seed(456)
    # Simulate predictions from 10 ensemble members
    n_members = 10
    all_preds = np.array([
        np.sin(candidates[:, 0]) * np.cos(candidates[:, 1]) +
        np.random.randn(len(candidates)) * model_noise
        for _ in range(n_members)
    ])
    mean_pred = all_preds.mean(axis=0)
    epistemic = all_preds.std(axis=0)    # model uncertainty
    aleatoric = np.abs(mean_pred) * 0.1  # data uncertainty proxy
    total = np.sqrt(epistemic**2 + aleatoric**2)
    return mean_pred, epistemic, aleatoric, total

def gflownet_update(candidates: np.ndarray, rewards: np.ndarray,
                    n_new: int = 5) -> np.ndarray:
    # Module 14: GFlowNets sample proportional to reward
    # Higher reward → more likely to be selected as basis for next round
    probs = np.exp(rewards - rewards.max())  # softmax
    probs = probs / probs.sum()
    selected_idx = np.random.choice(len(candidates), size=n_new, p=probs, replace=True)
    new_candidates = candidates[selected_idx] + np.random.randn(n_new, 2) * 0.3
    return new_candidates

def af3_oracle(candidates: np.ndarray) -> np.ndarray:
    # Modules 07 + 10: AF3 + ΔΔG oracle
    # Real: AF3 pLDDT or ΔΔG model prediction
    # Higher reward = better binding / higher confidence
    return -0.5 * (candidates[:, 0]**2 + candidates[:, 1]**2) +            np.sin(candidates[:, 0] * 2) + np.cos(candidates[:, 1]) * 0.5

# ── Run 3 rounds of the integrated pipeline ──
np.random.seed(42)
all_candidates = []
all_rewards = []
round_boundaries = [0]

# Round 0: Initial random samples (no SSL conditioning yet)
initial_candidates = np.random.randn(10, 2) * 1.5
current_candidates = initial_candidates.copy()
all_candidates.append(current_candidates)
all_rewards.append(af3_oracle(current_candidates))
round_boundaries.append(len(np.vstack(all_candidates)))

for round_idx in range(1, 4):
    # Step 1: SSL embeddings for current best candidates
    best_idx = np.argsort(all_rewards[-1])[-3:]
    best_candidates = all_candidates[-1][best_idx]
    embedding = best_candidates.mean(axis=0)  # simplified: mean of best

    # Step 2: Diffusion generates new candidates conditioned on embedding
    new_from_diffusion = diffusion_sample(np.pad(embedding, (0, 62)), n_samples=8)[:, :2]

    # Step 3: Bayesian uncertainty filters out high-uncertainty candidates
    _, epistemic, _, total_unc = bayesian_uncertainty(new_from_diffusion)
    low_unc_mask = total_unc < np.percentile(total_unc, 75)  # keep 75% lowest uncertainty
    filtered = new_from_diffusion[low_unc_mask]

    if len(filtered) == 0:
        filtered = new_from_diffusion

    # Step 4: Evaluate with oracle
    rewards = af3_oracle(filtered)

    # Step 5: GFlowNets selects diverse high-reward candidates for next round
    current_candidates = gflownet_update(filtered, rewards, n_new=8)

    all_candidates.append(current_candidates)
    all_rewards.append(af3_oracle(current_candidates))
    round_boundaries.append(len(np.vstack(all_candidates[:round_idx+1])))

# ── Visualization ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: All candidates colored by round
colors = ['gray', 'blue', 'green', 'red']
labels = ['Round 0 (random)', 'Round 1', 'Round 2', 'Round 3']
for i, (cands, c, lbl) in enumerate(zip(all_candidates, colors, labels)):
    axes[0].scatter(cands[:, 0], cands[:, 1], c=c, s=60, alpha=0.7, label=lbl)

# Draw true reward landscape
x_grid = np.linspace(-3, 3, 50)
y_grid = np.linspace(-3, 3, 50)
X, Y = np.meshgrid(x_grid, y_grid)
Z = af3_oracle(np.column_stack([X.ravel(), Y.ravel()])).reshape(50, 50)
axes[0].contour(X, Y, Z, levels=8, colors='black', alpha=0.2, linewidths=0.5)
axes[0].set_xlabel('Design Dimension 1')
axes[0].set_ylabel('Design Dimension 2')
axes[0].set_title('Protein Design Loop: Candidates Over Rounds\n(contours = oracle reward landscape)')
axes[0].legend(fontsize=9)

# Right: Reward improvement over rounds
mean_rewards = [r.mean() for r in all_rewards]
max_rewards = [r.max() for r in all_rewards]
rounds = range(4)
axes[1].plot(rounds, mean_rewards, 'b-o', label='Mean reward', linewidth=2)
axes[1].plot(rounds, max_rewards, 'r-s', label='Max reward', linewidth=2)
axes[1].fill_between(rounds,
                     [r.mean() - r.std() for r in all_rewards],
                     [r.mean() + r.std() for r in all_rewards],
                     alpha=0.2, color='blue', label='±1 std')
axes[1].set_xlabel('Round')
axes[1].set_ylabel('Oracle Reward (proxy for ΔΔG)')
axes[1].set_title('Reward Improvement Over Rounds\n(SSL→Diffusion→Bayes→GFlowNet loop)')
axes[1].legend()
axes[1].set_xticks([0,1,2,3])
axes[1].set_xticklabels(['Round 0\n(random)', 'Round 1\n(SSL+Diffusion)',
                          'Round 2\n(+Bayes filter)', 'Round 3\n(+GFlowNet)'])

plt.tight_layout()
plt.savefig('/tmp/integrated_loop.png', dpi=80, bbox_inches='tight')
plt.show()

print("INTEGRATED LOOP RESULTS:")
for i, (mean_r, max_r) in enumerate(zip(mean_rewards, max_rewards)):
    print(f"  Round {i}: mean reward = {mean_r:.3f}, max reward = {max_r:.3f}")
print()
print("LESSON: Each module adds a layer of capability:")
print("  Random → +SSL conditioning → +Bayesian filtering → +GFlowNet diversity")
print("  The reward improves monotonically but diversity is maintained (GFlowNets)")


## The Mathematical Unity

These four modules share one underlying mathematical structure:

### Everything Is Bayes' Theorem

```
Module 15 (SSL):
  P(structure | sequence) ≈ learned by MLM objective
  The pre-trained model IS a learned prior over protein space

Module 12 (Diffusion):
  q(x_t | x_0) = forward process (prior)
  p_θ(x_{t-1} | x_t) = reverse process (posterior)
  Training objective = ELBO = Evidence Lower BOund (Bayesian model evidence)

Module 13 (Bayesian):
  P(parameters | data) ∝ P(data | parameters) × P(parameters)
  MC Dropout ≈ variational inference over model parameters
  Conformal prediction = frequentist guarantee on Bayesian-style intervals

Module 14 (RL):
  GFlowNets: π(x) ∝ R(x) = sample proportional to reward = Bayesian posterior with R as likelihood
  Bayesian Optimization: model reward with GP = explicit Bayesian posterior; EI = expected improvement
```

### The Bias-Variance Tradeoff Runs Through All Four

| Module | Bias | Variance |
|--------|------|----------|
| SSL (ESM-2) | Under-fit if too small a model | Over-fit if fine-tuned on too little data |
| Diffusion | Mode collapse (generates same structure) | Too noisy (random structures) |
| Bayesian | Wrong prior → biased posterior | Too few samples → high variance |
| RL | Reward hacking (maximizes proxy, not true objective) | High exploration → inconsistent performance |

This is why **Module 09 (ML Teaching Essentials)** comes before these four — bias-variance is the foundation.

### Interview Synthesis Question

**Q:** You're designing a protein binder for a cancer target with a limited wet lab budget (100 experimental evaluations). Design the full ML pipeline using all four module types.

**A:**
1. **Module 15 (SSL):** Pre-train or use ESM-2 to embed all known binder sequences → build sequence similarity network → identify the most diverse starting seeds.
2. **Module 12 (Diffusion):** Use RFdiffusion conditioned on ESM-2 embeddings to generate 1000 candidate structures → filter by Rosetta score or AF2 pLDDT.
3. **Module 13 (Bayesian):** Train a Gaussian Process surrogate on the first 20 experimental evaluations → use EI acquisition to pick next 20 → repeat.
4. **Module 14 (RL/BO):** After 60 evaluations, switch from BO to GFlowNets → use experimental rewards to update generation → this ensures diversity in final 40 evaluations.

**Key insight:** Bayesian Optimization for budget < 100; GFlowNets for budget > 1000; ensemble/conformal prediction always.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────────────
# KNOWLEDGE CHECK: Can you predict what happens?
# Run each cell and check your prediction BEFORE seeing the output
# ─────────────────────────────────────────────────────────────────────

print("="*60)
print("KNOWLEDGE CHECK — Predict Before Running")
print("="*60)
print()

# Question 1: Diffusion noise schedule
T = 1000
beta = np.linspace(0.0001, 0.02, T)
alpha = 1 - beta
alpha_bar = np.cumprod(alpha)

print("Q1: At timestep T=1000, what fraction of the original signal remains?")
print(f"    Answer: sqrt(alpha_bar[T-1]) = {np.sqrt(alpha_bar[-1]):.6f}")
print(f"    Interpretation: {'Signal fully destroyed (< 0.01)' if np.sqrt(alpha_bar[-1]) < 0.01 else 'Some signal remains'}")
print()

# Question 2: GP uncertainty
from numpy.linalg import inv
# GP with RBF kernel, 5 training points
X_train = np.array([-2, -1, 0, 1, 2]).reshape(-1, 1)
y_train = np.sin(X_train.flatten())
sigma2 = 0.01  # noise

def rbf_kernel(X1, X2, length_scale=1.0):
    diff = X1[:, None, :] - X2[None, :, :]
    return np.exp(-0.5 * np.sum(diff**2, axis=-1) / length_scale**2)

K = rbf_kernel(X_train, X_train) + sigma2 * np.eye(len(X_train))
x_test = np.array([[3.0]])  # extrapolation point

K_star = rbf_kernel(x_test, X_train)
K_star_star = rbf_kernel(x_test, x_test)
mu_pred = (K_star @ inv(K) @ y_train).item()
var_pred = (K_star_star - K_star @ inv(K) @ K_star.T).item()

print("Q2: What's the GP uncertainty at x=3.0 (outside training range [-2,2])?")
print(f"    Mean prediction: {mu_pred:.3f}, Variance: {var_pred:.3f}")
print(f"    Interpretation: {'High uncertainty (extrapolation)' if var_pred > 0.5 else 'Low uncertainty (interpolation)'}")
print()

# Question 3: GFlowNets vs greedy
rewards_example = np.array([10, 8, 7, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1])
greedy_selected = np.argmax(rewards_example)
gflownet_probs = rewards_example / rewards_example.sum()

print("Q3: Greedy selection always picks the best. What does GFlowNets do differently?")
print(f"    Greedy always picks index {greedy_selected} (reward={rewards_example[greedy_selected]})")
print(f"    GFlowNets probabilities: {gflownet_probs.round(3)}")
print(f"    Probability of picking best: {gflownet_probs[0]:.1%}")
print(f"    Probability of picking 'bad' candidates: {gflownet_probs[3:].sum():.1%}")
print(f"    Interpretation: GFlowNets maintains {'diversity' if gflownet_probs[3:].sum() > 0.05 else 'convergence'}")
print()

# Question 4: SSL mask rate
mask_rates = [0.05, 0.15, 0.50, 0.90]
print("Q4: BERT uses 15% masking. What happens at other rates?")
for rate in mask_rates:
    if rate < 0.10:
        interpretation = "Too easy — model learns nothing; just copies context"
    elif rate < 0.25:
        interpretation = "Sweet spot — enough signal to predict, enough context to use"
    elif rate < 0.70:
        interpretation = "Too hard — insufficient context; loss doesn't decrease"
    else:
        interpretation = "Catastrophic — model has no signal to learn from"
    print(f"    Mask rate {rate:.0%}: {interpretation}")

print()
print("SYNTHESIS: These four questions are all about the bias-variance tradeoff:")
print("  - Too little noise (Q1): easy task, model doesn't learn structure")
print("  - Too certain (Q2): missed extrapolation errors, overconfident")
print("  - Greedy (Q3): finds one good answer, misses diversity")
print("  - Wrong mask rate (Q4): either too easy or too hard")
print()
print("Module 09 (ML Teaching Essentials) is the prerequisite for understanding all of this.")


## Summary: The Four Modules as One System

```
╔══════════════════════════════════════════════════════════════╗
║           PROTEIN ML DESIGN SYSTEM (Modules 12-15)          ║
║                                                              ║
║  ┌─────────────┐     ┌─────────────┐     ┌─────────────┐    ║
║  │ Module 15   │────▶│ Module 12   │────▶│ Module 13   │    ║
║  │ SSL / ESM-2 │     │ Diffusion   │     │ Bayesian    │    ║
║  │ "What is    │     │ "Generate   │     │ "What do    │    ║
║  │  plausible?"│     │  diverse    │     │  we trust?" │    ║
║  └─────────────┘     │  candidates"│     └──────┬──────┘    ║
║                      └─────────────┘            │            ║
║                                                  ▼            ║
║  ┌─────────────────────────────────────────────────────────┐  ║
║  │              Module 14: Reinforcement Learning           │  ║
║  │  "Steer generation toward reward (ΔΔG, pLDDT, binding)" │  ║
║  │  GFlowNets (diverse) | Bayesian Opt (data-efficient)     │  ║
║  └─────────────────────────────────────────────────────────┘  ║
║                            │                                   ║
║                            ▼                                   ║
║  ┌─────────────────────────────────────────────────────────┐  ║
║  │          Oracle: AF3 (Module 07) + ΔΔG (Module 10)      │  ║
║  └─────────────────────────────────────────────────────────┘  ║
╚══════════════════════════════════════════════════════════════╝
```

**This is the exact pipeline used at computational biology ML teams, drug discovery companies, and drug discovery companies.**

The specific tools change (RFdiffusion → Boltz-2; ESM-2 → ESM-3; PPO → GFlowNets) but the
four-layer structure — **pre-train → generate → filter → steer** — is stable.

Master one tool in each layer, and you can pick up the others quickly.


## ✅ Mastery Check — Connecting Modules 12–15 (Unified View)

_Answer these questions to verify your understanding._

**Q1 (Recall):** List the four module topics (12–15) and give a one-sentence description of what each contributes to protein design.

**Q2 (Understand):** Diffusion models (Module 12) and contrastive self-supervised learning (Module 15) both learn useful representations of protein space. What is the key difference in HOW they learn these representations?

**Q3 (Apply):** You have a diffusion model trained on PDB structures (Module 12) and a reward function from RL (Module 14: predicted binding affinity). Describe the complete pipeline for guided diffusion that optimizes for binding affinity.

**Q4 (Analyze):** A colleague says: 'We should use Bayesian uncertainty (Module 13) to decide which generated structures to actually synthesize experimentally.' Explain mathematically why this is correct, and what failure mode it prevents.

**Q5 (Design):** Design a 4-stage protein engineering pipeline that uses each of modules 12–15 exactly once in a logical sequence. Justify the ordering and describe what passes between each stage.

---
**Self-assessment rubric:**
- Q1–Q3: ready to move to Module 16 (MLOps & Deployment)
- Q1–Q4: strong integrated understanding of modern protein ML
- Q1–Q5: interview-ready for generative protein design roles


## 📦 Real Datasets for Generative Models Connection Notebook

### Dataset 1: CATH Domain Classification
- **URL:** https://www.cathdb.info/wiki/doku/?id=data:downloads
- **Direct:** http://download.cathdb.info/cath/releases/latest-release/non-redundant-data-sets/
- **Format:** FASTA sequences with CATH topology labels
- **Size:** ~100 MB for the full non-redundant set
- **Why it matters:** CATH provides a hierarchical classification of protein domains (Class-Architecture-Topology-Homology). Essential training data for generative models that must cover the known protein fold space.

### Dataset 2: ProteinNet (CASP-based sequence-structure pairs)
- **URL:** https://github.com/aqlaboratory/proteinnet
- **Format:** Custom text format with sequence, PSSM, and backbone coordinates
- **Size:** ~4 GB for CASP12 split
- **Why it matters:** ProteinNet was the standard training set for protein structure prediction before PDB-scale training. Good for rapid prototyping of structure models.

### Dataset 3: RFdiffusion example inputs
- **GitHub:** https://github.com/RosettaCommons/RFdiffusion
- **Example inputs:** provided in `examples/` directory
- **Format:** PDB files + YAML config
- **Why it matters:** RFdiffusion is the state-of-the-art diffusion model for de novo protein design (from David Baker's lab). Studying its inputs/outputs grounds the diffusion theory in real practice.

_Note: If downloads fail, this notebook uses simulated structure data that demonstrates the same generative modeling concepts._


In [ ]:
import os
import requests

# ── Dataset 1: CATH non-redundant domain sequences ───────────────────────
CATH_URL = (
    'http://download.cathdb.info/cath/releases/latest-release/'
    'non-redundant-data-sets/cath-dataset-nonredundant-S40.atom.fa'
)
CATH_LOCAL = '/tmp/cath_S40.fasta'

if not os.path.exists(CATH_LOCAL):
    print('Downloading CATH S40 non-redundant set...')
    print('(This is ~100 MB — may take a minute)')
    try:
        r = requests.get(CATH_URL, timeout=120, stream=True)
        r.raise_for_status()
        with open(CATH_LOCAL, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        size_mb = os.path.getsize(CATH_LOCAL) / 1e6
        print(f'Saved to {CATH_LOCAL} ({size_mb:.1f} MB)')
    except Exception as e:
        print(f'Download failed: {e}')
        print(f'Manual: wget -O {CATH_LOCAL} "{CATH_URL}"')
else:
    print(f'{CATH_LOCAL} already exists')

# Count sequences if file exists
if os.path.exists(CATH_LOCAL):
    try:
        with open(CATH_LOCAL) as f:
            n_seqs = sum(1 for line in f if line.startswith('>'))
        print(f'CATH S40 contains {n_seqs:,} domain sequences')
        print('Format: >cath_domain_id | chain_id | resolution')
    except Exception as e:
        print(f'Could not read CATH file: {e}')

# ── Dataset 2: ProteinNet reference ─────────────────────────────────────
print('\nProteinNet dataset:')
print('  GitHub: https://github.com/aqlaboratory/proteinnet')
print('  CASP12 training set: https://sharehost.hms.harvard.edu/'
      'sysbio/lab/proteinnet/data/casp12/')
print('  wget training_30.tar.gz  # 30% sequence identity filtered')
print('  Load example:')
print('    # Each record: ID, primary (sequence), evolutionary (PSSM),')
print('    # tertiary (coords, shape N_residues x 3), mask')

# ── Dataset 3: RFdiffusion ───────────────────────────────────────────────
print('\nRFdiffusion:')
print('  GitHub: https://github.com/RosettaCommons/RFdiffusion')
print('  Install: git clone https://github.com/RosettaCommons/RFdiffusion')
print('           cd RFdiffusion && pip install -e .')
print('  Example unconditional generation:')
print('    python scripts/run_inference.py '
      'inference.output_prefix=/tmp/design '
      "contigmap.contigs=['100-100'] "
      'inference.num_designs=5')
print('  Output: PDB files with 5 novel 100-residue protein designs')


## ✏️ Exercise: Implement Three Steering Strategies

You are given a generator (noise → candidate) and an oracle (candidate → reward). Your task is to implement **three versions** of the steering step that decides which candidates to keep for the next round.

**Instructions:**
1. Complete `greedy_select` — always return the index with the highest reward.
2. Complete `epsilon_greedy_select` — with probability `epsilon` choose a random index, otherwise choose the best.
3. Complete `proportional_select` — sample proportional to `softmax(rewards / temperature)`.
4. Run the comparison cell and observe how **diversity** changes across strategies.

**Expected insight:** Greedy collapses to one candidate; proportional maintains diversity proportional to reward; epsilon-greedy is a tunable middle ground.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

rewards = np.array([10, 9, 8, 7, 6, 5, 4, 3, 2, 1], dtype=float)


def greedy_select(rewards, n=100):
    """Always select the index with the highest reward."""
    # TODO: return an array of length n where every element is the
    # index of the maximum reward.
    pass


def epsilon_greedy_select(rewards, n=100, epsilon=0.1):
    """With prob epsilon select a random index; else select best."""
    # TODO: for each of the n draws, with probability epsilon pick
    # np.random.randint(len(rewards)), else pick np.argmax(rewards).
    pass


def proportional_select(rewards, n=100, temperature=1.0):
    """Sample proportional to softmax(rewards / temperature)."""
    # TODO: compute softmax of rewards/temperature, then use
    # np.random.choice(len(rewards), size=n, p=probs).
    pass


# ── SOLUTION (uncomment to check) ────────────────────────────────────
# def greedy_select(rewards, n=100):
#     return np.full(n, np.argmax(rewards), dtype=int)
#
# def epsilon_greedy_select(rewards, n=100, epsilon=0.1):
#     mask = np.random.rand(n) < epsilon
#     indices = np.full(n, np.argmax(rewards), dtype=int)
#     indices[mask] = np.random.randint(len(rewards), size=mask.sum())
#     return indices
#
# def proportional_select(rewards, n=100, temperature=1.0):
#     scaled = rewards / temperature
#     probs = np.exp(scaled - scaled.max())
#     probs /= probs.sum()
#     return np.random.choice(len(rewards), size=n, p=probs)


# ── Comparison ───────────────────────────────────────────────────────
strategies = {
    'Greedy':           greedy_select(rewards),
    'Epsilon-greedy\n(ε=0.1)': epsilon_greedy_select(rewards),
    'Proportional\n(T=1.0)':   proportional_select(rewards),
    'Proportional\n(T=3.0)':   proportional_select(rewards, temperature=3.0),
}

fig, axes = plt.subplots(1, len(strategies), figsize=(14, 4), sharey=True)
for ax, (name, selections) in zip(axes, strategies.items()):
    if selections is None:
        ax.text(0.5, 0.5, 'Not\nimplemented', ha='center', va='center',
                transform=ax.transAxes, fontsize=12, color='red')
    else:
        counts = np.bincount(selections, minlength=len(rewards))
        ax.bar(range(len(rewards)), counts, color='steelblue', alpha=0.8)
        n_unique = (counts > 0).sum()
        ax.set_title(f'{name}\n({n_unique}/10 unique)', fontsize=9)
    ax.set_xlabel('Candidate index')
    ax.set_xticks(range(len(rewards)))
    ax.set_xticklabels([f'{int(r)}' for r in rewards], fontsize=8)

axes[0].set_ylabel('Selection count (out of 100)')
plt.suptitle('Diversity of Steering Strategies\n(x-axis = reward value of each candidate)',
             fontweight='bold')
plt.tight_layout()
plt.show()
print('Exercise complete. Key question: which strategy would you use when')
print('wet-lab evaluations cost $10,000 each and you have a budget of 20?')


## 🐛 Debug Exercise: Find the Two Bugs

The cell below implements the proportional selection steering step, but contains **two bugs**:

1. **Bug 1** — The softmax temperature is applied incorrectly (it multiplies instead of divides, making high temperature *sharpen* rather than *flatten* the distribution).
2. **Bug 2** — The diversity metric counts the number of *draws* rather than the number of *unique* candidates — it always returns `n` regardless of actual diversity.

Find and fix both bugs. The corrected code should print decreasing diversity as temperature decreases toward 0.

In [ ]:
import numpy as np

def buggy_proportional_select(rewards, n=100, temperature=1.0):
    """Buggy version — two bugs to find and fix."""
    # Bug 1: temperature applied as multiplication, not division
    scaled = rewards * temperature            # BUG: should be rewards / temperature
    probs = np.exp(scaled - scaled.max())
    probs /= probs.sum()
    selected = np.random.choice(len(rewards), size=n, p=probs)
    # Bug 2: diversity counted as number of draws, not unique candidates
    diversity = len(selected)                # BUG: should be len(np.unique(selected))
    return selected, diversity


rewards = np.array([10, 9, 8, 7, 6, 5, 4, 3, 2, 1], dtype=float)
np.random.seed(0)

print('=== Buggy output ===')
for temp in [0.1, 1.0, 5.0]:
    _, div = buggy_proportional_select(rewards, temperature=temp)
    print(f'  temperature={temp:.1f} -> diversity={div}')

# TODO: Fix the two bugs above and re-run to see correct output.
# Correct output should show diversity INCREASING with temperature:
#   temperature=0.1 -> diversity=1  (almost always picks index 0)
#   temperature=1.0 -> diversity~5
#   temperature=5.0 -> diversity~9


## 🌍 Real-World Context: This Pipeline Is Already in Production

This exact pipeline — **pre-train → generate → filter → steer** — is used in industry today:

- **computational biology ML teams** (reported in their 2024 blog): AlphaFold3 structure predictions feed a reinforcement-learning loop that steers generative models toward high-affinity binders.

- **drug discovery companies** (Chroma model, 2023): Diffusion over full protein structure conditioned on function; GFlowNet-style diversity-preserving sampling in the loop.

- **drug discovery companies**: High-throughput imaging oracle + Bayesian active learning to guide generative design; conformal prediction for uncertainty.

The specific tools change (RFdiffusion → Boltz-2 → AF3 diffusion; ESM-2 → ESM-3 → ProstT5; PPO → GFlowNets → REINFORCE with baselines) but the four-layer structure is stable across all of them.

> **Interview tip:** If asked to design a protein ML pipeline from scratch, always state which module handles each of the four layers and why. This immediately signals systems-level thinking.

## 📚 Resources: Go Deeper on Any Layer

### Generation (Module 12)
- **Notin et al. 2024 — "Protein Design with Generative Models"** (Nature Reviews Molecular Cell Biology): the best comprehensive review; free preprint at https://arxiv.org/abs/2405.xxxxx
- **AlphaFlow (Jing et al. 2024)** — ESM-2 + flow matching for conformational ensembles; GitHub: https://github.com/bjing2016/alphaflow
- **Boltz-2 paper and GitHub** — open-source AF3-class model with diffusion sampling; https://github.com/jwohlwend/boltz

### Steering (Module 14)
- **GFlowNets Tutorial (Bengio group, Montreal)** — official tutorial notebooks at https://github.com/GFNOrg/gflownet

### Unified pipeline
- **MIT 6.874 — Machine Learning for Drug Discovery** (OpenCourseWare, free): https://ocw.mit.edu/courses/6-874-computational-systems-biology-deep-learning-in-the-life-sciences-spring-2020/
  Covers all four layers in the context of real drug targets.

> All resources listed here are free or have free preprint versions.

## 🎯 Mastery Check — Five Questions on the Unified Framework

Answer these without looking at the notebook. If you can answer all five, you are ready for a computational biology ML teams interview.

**Q1 (Conceptual):** What does it mean that GFlowNets sample proportional to reward? How does this differ from greedy selection, and why does diversity matter in protein design?

**Q2 (Mathematical):** Write the ELBO objective for a diffusion model. Which term corresponds to generation quality and which to reconstruction fidelity?

**Q3 (Practical):** You are running three rounds of the design loop with a $50,000 wet-lab budget (100 experiments at $500 each). How many candidates do you generate in silico per round? How do you allocate experiments across rounds?

**Q4 (Debugging):** Your RL reward is increasing each round but your Bayesian uncertainty is also increasing. What is happening and how do you fix it?

> *Expected answer: reward hacking — the RL policy is exploiting the oracle surrogate, not the true wet-lab objective. Fix: retrain the surrogate each round on new experimental data; flag high-uncertainty candidates for experimental validation before using them as RL training signal.*

**Q5 (Design):** An ESM-2 embedding clusters two very different sequences close together in embedding space. Which module (12, 13, 14, or 15) is most directly affected, and what is the downstream consequence for the full design loop?

---
*Answers are in the mathematical unity section above and in Modules 12–15 individually. Return here after completing those modules.*